Wide&deep

In [42]:
!pip install -q wandb

In [43]:
import wandb

In [46]:
wandb.login()

True

In [37]:
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import wandb

In [23]:
df = pd.read_csv("weekly_driving_profiles.csv")

df.head()

,driver_id,week,engine_capacity,road_quality_moderate,slope_flat,motorway,rural,more_than_one_lane,clear_weather,congested,...,harsh_acceleration,harsh_braking,forward_collision,distracted_driver,too_close_distance,lane_departure,driver_making_calls,driver_smoking,fatigue_driving,claims_count
0,0,2021-06-13 00:00:00+00:00,2.299,0.497795,0.661735,0.159487,0.354314,0.223157,0.491282,0.004252,...,0.0,0.0,7.0,212.0,0.0,6.0,0.0,0.0,0.0,0.0
1,0,2021-06-20 00:00:00+00:00,2.299,0.446922,0.682769,0.251670,0.342359,0.291571,0.695668,0.000687,...,0.0,0.0,8.0,593.0,0.0,144.0,2.0,0.0,0.0,0.0
2,0,2021-06-27 00:00:00+00:00,2.299,0.403515,0.792455,0.386323,0.266479,0.477638,0.435115,0.000295,...,0.0,0.0,6.0,75.0,26.0,308.0,3.0,4.0,0.0,0.0
3,0,2021-07-04 00:00:00+00:00,2.299,0.504037,0.972110,0.347816,0.394169,0.521672,1.000000,0.002024,...,0.0,0.0,1.0,1.0,1.0,104.0,8.0,0.0,0.0,0.0
4,0,2021-07-11 00:00:00+00:00,2.299,0.360000,0.962667,0.712000,0.080000,0.829333,1.000000,0.000000,...,0.0,0.0,1.0,0.0,7.0,42.0,0.0,1.0,0.0,0.0


In [24]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

Настройки модели

In [25]:
class WideDeepCFG:
    seed = 30

    # Задаем параметры нашего эксперимента
    feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

    target_col = 'claims_count'

    input_dim = len(feature_cols)
    hidden_dims = [64, 32]
    dropout_rate = 0.3
    batch_size = 64
    learning_rate = 0.0005
    num_epochs = 60

X - признаки, по которым модель учится.

y - целевая переменная, которую модель предсказывает

Train / validation / test split

In [26]:
seed_everything(WideDeepCFG.seed)

feature_cols = WideDeepCFG.feature_cols
target_col = WideDeepCFG.target_col

unique_drivers = df["driver_id"].unique()

train_drivers, temp_drivers = train_test_split(
    unique_drivers,
    test_size=0.3,
    random_state=WideDeepCFG.seed
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.5,
    random_state=WideDeepCFG.seed
)

X_train = df[df["driver_id"].isin(train_drivers)][feature_cols]
y_train = df[df["driver_id"].isin(train_drivers)][target_col]

X_val = df[df["driver_id"].isin(val_drivers)][feature_cols]
y_val = df[df["driver_id"].isin(val_drivers)][target_col]

X_test = df[df["driver_id"].isin(test_drivers)][feature_cols]
y_test = df[df["driver_id"].isin(test_drivers)][target_col]

print("Train drivers:", len(train_drivers), "rows:", len(X_train))
print("Val drivers:", len(val_drivers), "rows:", len(X_val))
print("Test drivers:", len(test_drivers), "rows:", len(X_test))

Train drivers: 247 rows: 8465
Val drivers: 53 rows: 2027
Test drivers: 54 rows: 2036


Масштабирование признаков

In [27]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

DataLoader

In [28]:
def make_loader(X_data, y_data, batch_size, shuffle, drop_last=False):
    X_tensor = torch.tensor(X_data, dtype=torch.float32)
    y_tensor = torch.tensor(y_data.values, dtype=torch.float32).reshape(-1, 1)
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

    return loader


train_loader = make_loader(X_train_scaled, y_train, WideDeepCFG.batch_size, shuffle=True, drop_last=True)

val_loader = make_loader(X_val_scaled, y_val, WideDeepCFG.batch_size, shuffle=False)

test_loader = make_loader(X_test_scaled, y_test, WideDeepCFG.batch_size, shuffle=False)

Архитектура Wide&Deep

In [29]:
class WideAndDeepRegression(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout_rate):
        super(WideAndDeepRegression, self).__init__()

        self.wide = nn.Linear(input_dim, 1)

        deep_layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            deep_layers.append(nn.Linear(prev_dim, hidden_dim))
            deep_layers.append(nn.BatchNorm1d(hidden_dim))
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.Dropout(dropout_rate))

            prev_dim = hidden_dim

        deep_layers.append(nn.Linear(prev_dim, 1))

        self.deep = nn.Sequential(*deep_layers)

        self.output_activation = nn.Softplus()

    def forward(self, x):
        wide_output = self.wide(x)
        deep_output = self.deep(x)

        output = wide_output + deep_output

        output = self.output_activation(output)

        return output

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = WideAndDeepRegression(
    input_dim=WideDeepCFG.input_dim,
    hidden_dims=WideDeepCFG.hidden_dims,
    dropout_rate=WideDeepCFG.dropout_rate
)

model = model.to(device)

criterion = nn.PoissonNLLLoss(log_input=False)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=WideDeepCFG.learning_rate
)

print(model)

WideAndDeepRegression(
  (wide): Linear(in_features=19, out_features=1, bias=True)
  (deep): Sequential(
    (0): Linear(in_features=19, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
  (output_activation): Softplus(beta=1.0, threshold=20.0)
)


In [51]:
run = wandb.init(
    entity="masksasha-hse-university",
    project="insurance-driving-and-damage",
    group="tabular_discount_prediction",
    name="wide_deep_poisson",
    config={
        "task": "tabular_claims_count_prediction",
        "model_type": "Wide & Deep",
        "target": WideDeepCFG.target_col,
        "split_type": "driver_id",
        "random_state": WideDeepCFG.seed,
        "input_dim": WideDeepCFG.input_dim,
        "hidden_dims": WideDeepCFG.hidden_dims,
        "dropout_rate": WideDeepCFG.dropout_rate,
        "batch_size": WideDeepCFG.batch_size,
        "learning_rate": WideDeepCFG.learning_rate,
        "num_epochs": WideDeepCFG.num_epochs,
        "loss_function": "PoissonNLLLoss",
        "optimizer": "Adam"
    }
)

функция обучения одной эпохи

In [52]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()

    loss_sum = 0
    n_objects = 0

    for data, target in train_loader:
        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = model(data)

        loss = criterion(output, target)

        loss.backward()

        optimizer.step()

        loss_sum += loss.item() * data.size(0)
        n_objects += data.size(0)

    train_poisson_loss = loss_sum / n_objects

    return train_poisson_loss

In [53]:
def evaluate(model, loader, criterion, device):
    model.eval()

    loss_sum = 0
    n_objects = 0

    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            target = target.to(device)

            output = model(data)

            loss = criterion(output, target)

            loss_sum += loss.item() * data.size(0)
            n_objects += data.size(0)

    poisson_loss = loss_sum / n_objects

    return poisson_loss

Обучение модели

In [54]:
best_val_loss = float("inf")
best_epoch = 0

history = []

for epoch in range(1, WideDeepCFG.num_epochs + 1):
    train_poisson_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_poisson_loss = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    history.append({
        "epoch": epoch,
        "train_poisson_loss": train_poisson_loss,
        "val_poisson_loss": val_poisson_loss
    })

    wandb.log({
        "train_poisson_loss": train_poisson_loss,
        "val_poisson_loss": val_poisson_loss
    }, step=epoch)

    if val_poisson_loss < best_val_loss:
        best_val_loss = val_poisson_loss
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            "wide_deep_best_model.pt"
        )

    print(
        "epoch:", epoch,
        "train poisson:", round(train_poisson_loss, 4),
        "val poisson:", round(val_poisson_loss, 4)
    )

print("Best epoch:", best_epoch)
print("Best validation poisson loss:", best_val_loss)

epoch: 1 train poisson: 0.5165 val poisson: 0.5372
epoch: 2 train poisson: 0.4505 val poisson: 0.522
epoch: 3 train poisson: 0.429 val poisson: 0.5273
epoch: 4 train poisson: 0.4158 val poisson: 0.5318
epoch: 5 train poisson: 0.4155 val poisson: 0.5377
epoch: 6 train poisson: 0.4115 val poisson: 0.5337
epoch: 7 train poisson: 0.4092 val poisson: 0.5361
epoch: 8 train poisson: 0.408 val poisson: 0.541
epoch: 9 train poisson: 0.4054 val poisson: 0.5425
epoch: 10 train poisson: 0.4026 val poisson: 0.5395
epoch: 11 train poisson: 0.3993 val poisson: 0.5419
epoch: 12 train poisson: 0.3952 val poisson: 0.5415
epoch: 13 train poisson: 0.3929 val poisson: 0.5495
epoch: 14 train poisson: 0.3989 val poisson: 0.5411
epoch: 15 train poisson: 0.3915 val poisson: 0.5424
epoch: 16 train poisson: 0.3901 val poisson: 0.5355
epoch: 17 train poisson: 0.3911 val poisson: 0.5485
epoch: 18 train poisson: 0.39 val poisson: 0.5402
epoch: 19 train poisson: 0.3878 val poisson: 0.5455
epoch: 20 train poisson: 0.

Финальная проверка на test

In [55]:
best_model = WideAndDeepRegression(
    input_dim=WideDeepCFG.input_dim,
    hidden_dims=WideDeepCFG.hidden_dims,
    dropout_rate=WideDeepCFG.dropout_rate
)

best_model.load_state_dict(
    torch.load("wide_deep_best_model.pt", map_location=device)
)

best_model = best_model.to(device)

test_poisson_loss = evaluate(
    best_model,
    test_loader,
    criterion,
    device
)

print("Test poisson loss:", test_poisson_loss)

wandb.log({
    "test_poisson_loss": test_poisson_loss
})

wandb.run.summary["test_poisson_loss"] = test_poisson_loss
wandb.run.summary["best_epoch"] = best_epoch
wandb.run.summary["best_val_poisson_loss"] = best_val_loss

Test poisson loss: 0.4484562198164889


In [56]:
wandb.finish()

best_epoch,▁
best_val_poisson_loss,▁
test_poisson_loss,▁
train_poisson_loss,█▅▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁
val_poisson_loss,▃▁▂▂▃▃▄▃▃▃▃▄▃▄▅▅▆▄▅▄▅▆▅▄▆▄▆▆▆▆▆▆▅▆▇▇▅▆▆█
best_epoch,2
best_val_poisson_loss,0.52196
test_poisson_loss,0.44846
train_poisson_loss,0.36684
val_poisson_loss,0.5786
